# MARS-LVIG HKairport_GNSS03 bag -> drone-style video + GPS log

Turns the ROS bag into what a real drone flight hands RTVIO: **`flight.mp4`** (the left camera, H.264) and **`gps.csv`** (standalone u-blox fixes, seconds from the video's first frame). Video time equals capture time - a frame the camera dropped is held, not skipped - so the GPS log lines up exactly.

The RTK ground truth (`ground_truth_rtk.csv`) and `session_meta.json` are kept next to the video, so a run on the video can be scored with `tools/eval_trajectory.py --session <this folder>`.

**Before running**
1. Accelerator: **None (CPU)** is enough - no GPU is used. **Internet: On** (clones the repo, installs `rosbags`).
2. **Add Input -> Your Work**: the `mars-lvig-hkairport-gnss03` dataset (the 9.09 GB bag).
3. Run all. Outputs land in `/kaggle/working/hkairport_gnss03_video/` (Output tab), or publish them as a dataset with the last cell.

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO_DIR = "/kaggle/working/Terraform"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch linux-dev https://github.com/Jola7898/Terraform.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!cd {REPO_DIR} && git log --oneline -1
RTVIO = os.path.join(REPO_DIR, "rtvio")

!pip install -q rosbags pyyaml
if not shutil.which("ffmpeg"):
    !apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -1
!cd {RTVIO} && python tests/test_marslvig_tools.py | tail -1

In [ ]:
hits = glob.glob("/kaggle/input/**/HKairport_GNSS03.bag", recursive=True)
assert hits, "HKairport_GNSS03.bag not found under /kaggle/input - attach the dataset via Add Input -> Your Work"
BAG = hits[0]
CALIB = os.path.join(RTVIO, "data", "marslvig", "HK_GNSS_airport_island.yaml")
print("bag  :", BAG, "(%.2f GB)" % (os.path.getsize(BAG) / 1e9), "\ncalib:", CALIB)

## Settings

- `STRIDE = 1` keeps the camera's full 10 frames/s; reconstruct with `--sample-fps 5` to match the stride-2 benchmark runs.
- `LONG_EDGE = 0` keeps the native 2448x2048; `1920` gives a smaller, DJI-like file (VGGT sees 518 px wide frames either way).
- `CRF` is H.264 quality: 18 is visually lossless, 23 is ffmpeg's default and noticeably smaller.
- `TRIM = True` keeps only the flight (RTK horizontal speed > 1 m/s, +-2 s): the same span the benchmark scores.

In [ ]:
STRIDE, LONG_EDGE, CRF, TRIM = 1, 0, 18, True

WORK = "/tmp/hkairport_gnss03_video_session"          # the JPEG frames (~GBs) stay out of /kaggle/working
OUT = "/kaggle/working/hkairport_gnss03_video"

## Convert (one pass over the bag, then the H.264 encode - several minutes on Kaggle's CPUs)

In [ ]:
cmd = [sys.executable, "tools/marslvig_to_session.py", "--bag", BAG, "--calib", CALIB, "--out", WORK,
       "--stride", str(STRIDE), "--video", "--video-long-edge", str(LONG_EDGE), "--video-crf", str(CRF)]
if not TRIM:
    cmd.append("--no-trim")
if not os.path.exists(os.path.join(WORK, "flight.mp4")):
    subprocess.run(cmd, check=True, cwd=RTVIO)

os.makedirs(OUT, exist_ok=True)
for name in ("flight.mp4", "gps.csv", "camera_intrinsics.json", "ground_truth_rtk.csv", "session_meta.json"):
    shutil.copy2(os.path.join(WORK, name), os.path.join(OUT, name))
    print("%-24s %10.1f MB" % (name, os.path.getsize(os.path.join(OUT, name)) / 1e6))

## Check: duration, frame rate and GPS coverage agree

In [ ]:
probe = json.loads(subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_packets",
                                   "-show_entries", "stream=width,height,r_frame_rate,nb_read_packets,duration",
                                   "-of", "json", os.path.join(OUT, "flight.mp4")],
                                  capture_output=True, text=True, check=True).stdout)["streams"][0]
meta = json.load(open(os.path.join(OUT, "session_meta.json")))
num, den = map(int, probe["r_frame_rate"].split("/"))
fps = num / den
n = int(probe["nb_read_packets"])
print("video : %dx%d, %.3f fps, %d frames = %.1f s" % (probe["width"], probe["height"], fps, n, n / fps))
print("bag   : %d frames kept over %.1f s (stride %d)" % (meta["frames"], meta["span_s"], meta["stride"]))
gps = list(__import__("csv").DictReader(open(os.path.join(OUT, "gps.csv"))))
ts = [float(r["timestamp_s"]) for r in gps]
print("gps   : %d fixes from %.1f s to %.1f s (video runs 0 to %.1f s)" % (len(ts), min(ts), max(ts), n / fps))
assert abs(n / fps - (meta["span_s"] + 1 / fps)) < 2 / fps, "video length does not match the capture span"

In [ ]:
# One frame from mid-flight, to eyeball the video without downloading it.
from IPython.display import Image, display
mid = n // (2 * fps)
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(mid), "-i", os.path.join(OUT, "flight.mp4"),
                "-frames:v", "1", "-vf", "scale=960:-2", "/tmp/preview.jpg"], check=True)
display(Image("/tmp/preview.jpg"))

## Use it

In the benchmark notebook, or anywhere RTVIO runs (paths relative to this folder):

```
python -m rtvio.vggt_reconstruct --video flight.mp4 --gps gps.csv --intrinsics camera_intrinsics.json \
    --gps-mode guided --sample-fps 5 --out run_video
python tools/eval_trajectory.py --session <this folder> --run run_video
```

## Optional: publish as a private Kaggle dataset

Saves re-running the conversion: attach `<you>/mars-lvig-hkairport-gnss03-video` as an input next time. MARS-LVIG is CC BY-NC-SA 4.0, and so is anything derived from it. Needs `KAGGLE_USERNAME` / `KAGGLE_KEY` in Add-ons -> Secrets.

In [ ]:
PUBLISH = False

if PUBLISH:
    !pip install -q -U kagglehub
    import kagglehub
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ[k] = secrets.get_secret(k).strip()   # a pasted secret easily carries a stray space
    user = kagglehub.whoami()["username"]
    with open(os.path.join(OUT, "README.md"), "w") as f:
        f.write("HKairport_GNSS03 from the MARS-LVIG dataset (HKU MaRS Lab) as flight.mp4 + gps.csv, made by "
                "rtvio/tools/marslvig_to_session.py --video (stride %d, long edge %s, crf %d).\n"
                "Licence: CC BY-NC-SA 4.0, as the source dataset.\n" % (STRIDE, LONG_EDGE or "native", CRF))
    kagglehub.dataset_upload("%s/mars-lvig-hkairport-gnss03-video" % user, OUT,
                             version_notes="stride %d, long edge %s, crf %d" % (STRIDE, LONG_EDGE or "native", CRF))